In [1]:
## IMPORT LIBRARIES ----------------------------------------------------------------------------------------------------
#https://github.com/nnc-ufmg/circadipy/blob/main/src/circadipy/analysis_examples/intellicage/intellicage_analysis.ipynb
%matplotlib qt

%reload_ext autoreload
%autoreload 3
from ranking_methods import rank_accuracy
from scipy.stats import rankdata
import sys                                                                                                              # Import sys to add paths to libraries                                                                                                           # Import re to work with regular expressions
import glob                                                                                                             # Import glob to read files                                                                                                   # Import numpy to work with arrays and make calculations                                                                                            # Import time to measure time
import os                                                                                                               # Import path to work with paths                
import pandas as pd
from ranking_methods import build_all_proxies, evaluate_proxies
import numpy as np
from scipy import stats
                                                                                      # Import pandas to work with dataframes
import warnings                                                                                                         # Import warnings to ignore warnings
warnings.filterwarnings('ignore')                                                                                       # Ignore warnings

## IMPORT CIRCADIPY ----------------------------------------------------------------------------------------------------

parent_path = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.append(parent_path)

## PCA Visualization - Dimensionality Reduction
from models import train_rf, train_gb, train_ridge, train_adaboost, train_extratrees, train_logistic
from summary import generate_summary_report
from util import  get_data_scaled, generate_features, generate_temporal_features, calculate_correlations, build_animal_protocols, get_sorted_animals_files, combine_all_features
from analysis_visualization import generate_methods_comparison, plot_animals_activity, plot_correlation, plot_pca_analysis, plot_cross_correlation, summary_visualization

In [2]:
data_folder = "./data"                              
individual_files = glob.glob(data_folder + "/unwrapped_data/**/*.txt", recursive=True)

ranks_male = {
    "animal_5": 5,
    "animal_12": 8,
    "animal_9": 6,
    "animal_13": 4,
    "animal_15": 1,
    "animal_3": 3,
    "animal_8": 7,
    "animal_11": 2
}



ranks_female = {
    "animal_21": 1,
    "animal_18": 8,
    "animal_19": 9,
    "animal_24": 6,
    "animal_22": 4,
    "animal_26": 7,
    "animal_23": 2,
    "animal_17": 5,
    "animal_16": 3
}

#usando sum para concatenar os protocolos gera resultado melhor que o last
#Se eu quiser ler os protocolos eu preciso do zt_0_time, neste caso estou usando 20horas.
#Tambem precisariamos definir o labels dict, neste caso o que utilizariamos?
zt_0_time = 20  #Para gerar o actograma é usado o zt dado de quando a luz é acesa, que será as 20 horas. 
labels_dict = {'cycle_types': ['LD'], 'test_labels': ['1_control_dl'], 'cycle_days': [1]}
ranks_animals = {"male": ranks_male, "female": ranks_female}
gender = "male"

ranks = ranks_animals[gender]
animals = [int(k.split("_")[1]) for k in ranks.keys()]
animals_files = get_sorted_animals_files(individual_files, animals)
animals_protocols, animals_by_day = build_animal_protocols(animals_files, zt_0_time=zt_0_time, labels_dict=labels_dict)



Animal 3: 18
Animal 5: 16
Animal 8: 18
Animal 9: 19
Animal 11: 20
Animal 12: 20
Animal 13: 20
Animal 15: 19
Animal2 3
Animal2 5
Animal2 8
Animal2 9
Animal2 11
Animal2 12
Animal2 9
Animal2 11
Animal2 12
Animal2 13
Animal2 15
Animal animal_3 Days found: 47
Animal animal_5 Days found: 47
Animal2 13
Animal2 15
Animal animal_3 Days found: 47
Animal animal_5 Days found: 47
Animal animal_8 Days found: 47
Animal animal_9 Days found: 47
Animal animal_11 Days found: 47
Animal animal_12 Days found: 47
Animal animal_13 Days found: 47
Animal animal_15 Days found: 47
Animal animal_8 Days found: 47
Animal animal_9 Days found: 47
Animal animal_11 Days found: 47
Animal animal_12 Days found: 47
Animal animal_13 Days found: 47
Animal animal_15 Days found: 47


##### Total activy per day tem a soma de todos os dias. Ë possivel ver isso claramente abrindo o arquivo collect data onde será visto a atividade por zt de 0.5 em 0.5, somando-se todos os valores da coluna o valor para cada dia será igual ao total day activity

In [3]:


output_path = f'./test_results_2026/basic_features_{gender}.csv'
features_df = generate_features(animals_protocols, ranks, output_path=output_path)


output_path = f'./test_results_2026/temporal_features_{gender}.csv'
temporal_df = generate_temporal_features(animals_protocols, ranks, output_path=output_path)

output_path = f'./test_results_2026/all_features_{gender}.csv'
all_features, feature_cols = combine_all_features(features_df, temporal_df, output_path=output_path)

# Derived additive features

X_scaled =  get_data_scaled(all_features, feature_cols)
y = all_features['actual_rank'].values

output_correlation = f'./test_results_2026/feature_correlations_{gender}.csv'
corr_df = calculate_correlations(all_features, feature_cols, output_path=output_correlation)


Saving features on ./test_results_2026/basic_features_male.csv
Saving temporal features on ./test_results_2026/temporal_features_male.csv
Saving all features on ./test_results_2026/all_features_male.csv
Saving correlation on ./test_results_2026/feature_correlations_male.csv


In [6]:
# k = number of top features used by top-k methods; best_feat_idx = which ranked feature to use as single best
i = 2 #2, 7 #bom female
indices = [7] # 7 female
i = 0 #2, 7 #bom male
indices = [3] # 7 male
best_combo = ('ibi_kurt', 'ibi_p90', 'high_activity_frac', 'power_12h')

best_feat = 'ibi_median'
named_combo = None

best_feat = None
#named_combo = ['high_activity_frac', 'activity_skew', 'max_median_ratio', 'activity_per_bout']

if best_feat is None:
    print("Best feat is none, computing")
    best_feat = corr_df[corr_df.index.isin(feature_cols)]['correlation'].abs().idxmax()

if named_combo is None:
    print("Named combo is none, using computed")
    named_combo = list(best_combo) if 'best_combo' in dir() else None
    
print("aui")
print(best_feat)

feature_rhos, proxies_raw = build_all_proxies(
    all_features, feature_cols, X_scaled,
    k=3, best_feat_idx=best_feat,
    named_combo=named_combo
)


print('Top feature correlations (Spearman):')
print(feature_rhos.head(10))

proxy_output_path = f"./test_results_2026/proxy_summary_{gender}.csv"
proxy_rows, proxy_summary = evaluate_proxies(proxies_raw, all_features, verbose=True, output_path=proxy_output_path)




Best feat is none, computing
Named combo is none, using computed
aui
power_8h
Top feature correlations (Spearman):
power_8h             0.952381
cosinor_amplitude    0.857143
ibi_p90              0.714286
mean_bout_len        0.690476
cosinor_mesor        0.619048
num_bouts_night     -0.610789
bout_rate_night     -0.610789
p95_bout_len         0.487950
bout_rate_day       -0.476190
num_bouts_day       -0.476190
dtype: float64
Score [ 191.11940215 2385.69715508 4974.49040619 4707.14456043 1168.59798195
 5940.33164714 3208.83697239  143.84118717] scores oriented [ 191.11940215 2385.69715508 4974.49040619 4707.14456043 1168.59798195
 5940.33164714 3208.83697239  143.84118717]

Best feature (power_8h):
  Order: ['animal_15', 'animal_3', 'animal_11', 'animal_5', 'animal_13', 'animal_9', 'animal_8', 'animal_12']
  Actual ranks: [1, 3, 2, 5, 4, 6, 7, 8]
  Assigned ordinal ranks: [2, 4, 7, 6, 3, 8, 5, 1]
  Accuracy: 50.0%  Within±1: 100.0%  Within±2: 100.0%
Score [ 1.89536509 -3.63485493  3.52

In [5]:
actual_ranks = all_features['actual_rank'].values
for name,score in proxies_raw.items():
    ord_idx = np.argsort(score)
    ordered_animals = all_features.iloc[ord_idx]['animal'].tolist()
    print(name)
    pred_rank = rankdata(score, method='ordinal')
    print(ordered_animals)
    print(pred_rank)
    print(actual_ranks)

    metrics = rank_accuracy(actual_ranks, pred_rank)
    print(metrics)
    print()

 


Best feature (power_8h)
['animal_15', 'animal_3', 'animal_11', 'animal_5', 'animal_13', 'animal_9', 'animal_8', 'animal_12']
[2 4 7 6 3 8 5 1]
[3 5 7 6 2 8 4 1]
{'accuracy': 0.5, 'within_1': 1.0, 'within_2': 1.0, 'mae': 0.5, 'rho': 0.9523809523809524}

Best combo (ibi_kurt + ibi_p90 + high_activity_frac + power_12h)
['animal_5', 'animal_11', 'animal_15', 'animal_13', 'animal_9', 'animal_3', 'animal_12', 'animal_8']
[6 1 8 5 2 7 4 3]
[3 5 7 6 2 8 4 1]
{'accuracy': 0.25, 'within_1': 0.625, 'within_2': 0.75, 'mae': 1.5, 'rho': 0.6190476190476191}

Sum top3 sign-aligned z
['animal_15', 'animal_11', 'animal_5', 'animal_3', 'animal_13', 'animal_9', 'animal_12', 'animal_8']
[4 3 8 6 2 7 5 1]
[3 5 7 6 2 8 4 1]
{'accuracy': 0.375, 'within_1': 0.875, 'within_2': 1.0, 'mae': 0.75, 'rho': 0.9047619047619048}

Rank product (geom mean) top3
['animal_15', 'animal_5', 'animal_11', 'animal_3', 'animal_13', 'animal_9', 'animal_12', 'animal_8']
[4 2 8 6 3 7 5 1]
[3 5 7 6 2 8 4 1]
{'accuracy': 0.25, 'with

In [5]:

proxy_metrics_df = pd.DataFrame(proxy_rows)
print("\nProxy metrics summary:\n", proxy_metrics_df)

proxy_metrics_path = "./test_results_2026/proxy_metrics.csv"
proxy_metrics_df.to_csv(proxy_metrics_path, index=False)
print(f"Saved proxy metrics to {proxy_metrics_path}")

#Also keep the earlier unsupervised summary for quick reference
print("\nUnsupervised proxies vs rank (Spearman rho, MAE):")
for row in proxy_rows:
    print(f"{row['proxy']:<18s} rho={row['rho']: .3f}, MAE={row['mae']: .2f}, tau={row['kendall_tau']: .3f}")


Proxy metrics summary:
                                            proxy       rho  kendall_tau   mae  \
0                      Best feature (ibi_median)  0.976190     0.928571  0.25   
1   Custom sum indices=[3] (hourly_entropy_norm)  0.690476     0.571429  1.50   
2                        Sum top3 sign-aligned z  0.928571     0.857143  0.50   
3                  Rank product (geom mean) top3  0.904762     0.785714  0.75   
4            Borda weighted by |rho| (all feats)       NaN          NaN   NaN   
5                  PC1 projection (unsupervised)  0.142857     0.000000  2.50   
6                   Min-rank top3 (conservative)  0.904762     0.785714  0.75   
7                 Seriation euclidean (opt leaf)  0.333333     0.142857  2.25   
8                  Seriation spearman (opt leaf)  0.357143     0.142857  2.25   
9                         Mean z across features  0.238095     0.142857  2.25   
10                      Median z across features  0.642857     0.571429  1.50   
11 

In [6]:
output_corr = f"./test_results_2026/feature_correlations_{gender}.html"
fig_corr = plot_correlation(corr_df, output_corr)

male_comparison_output = f"./test_results_2026/animal_comparison_all_{gender}.html"
fig_activity = plot_animals_activity(animals_protocols, ranks, output_path=male_comparison_output)

output_path = f"./test_results_2026/pca_visualization_{gender}.html"
fig_pca = plot_pca_analysis(all_features, feature_cols, output_path=output_path)

output_path = f"./test_results_2026/cross_correlation_matrix_{gender}.html"
corr_sync, pval_sync, fig_cross = plot_cross_correlation(animals_protocols, ranks, output_path=output_path)


animal_3 in ranks, adding to plot
animal_5 in ranks, adding to plot
animal_8 in ranks, adding to plot
animal_9 in ranks, adding to plot
animal_11 in ranks, adding to plot
animal_12 in ranks, adding to plot
animal_13 in ranks, adding to plot
animal_15 in ranks, adding to plot



PCA Explained Variance Ratio: [0.52981012 0.23180452 0.10476665 0.05763065]
Total Variance Explained: 92.40%



Average Activity Synchronization:
      animal  avg_correlation  actual_rank
7  animal_15         0.345721            1
4  animal_11         0.473312            2
0   animal_3         0.441269            3
6  animal_13         0.526797            4
1   animal_5         0.383780            5
3   animal_9         0.461666            6
2   animal_8         0.448019            7
5  animal_12         0.457305            8

Correlation between synchronization and rank: 0.190 (p=0.651)


In [7]:
output_path = "./test_results_2026/feature_importance_gb.html"
gb, gb_mae, all_features, fig_gb = train_gb(all_features, feature_cols, X_scaled, output_path=output_path)

Using top-10 features: ['min_activity', 'median_activity', 'ibi_median', 'bout_len_cv', 'day_night_ratio', 'daynight_log_ratio', 'day_cv_activity', 'short_bout_frac', 'max_activity', 'max_median_ratio']

Gradient boosting Feature Importance:
              feature  importance
7     short_bout_frac    0.179999
3         bout_len_cv    0.177244
4     day_night_ratio    0.146915
9    max_median_ratio    0.141132
8        max_activity    0.109040
2          ibi_median    0.100391
5  daynight_log_ratio    0.074878
6     day_cv_activity    0.070402
0        min_activity    0.000000
1     median_activity    0.000000



Leave-2-Out Cross-Validation:
Mean Absolute Error: 2.06 ranks
Std: 1.11

Actual vs Predicted Ranks:
              animal  actual_rank  gb_rank
animal_15  animal_15            1        1
animal_11  animal_11            2        2
animal_3    animal_3            3        3
animal_13  animal_13            4        4
animal_5    animal_5            5        5
animal_9    animal_9            6        6
animal_8    animal_8            7        7
animal_12  animal_12            8        8


In [8]:
output_path = "./test_results_2026/feature_importance_ridge.html"
ridge, ridge_mae, all_features, fig_ridge = train_ridge(all_features, feature_cols, X_scaled, output_path=output_path)

Using top-10 features: ['min_activity', 'median_activity', 'ibi_median', 'bout_len_cv', 'day_night_ratio', 'daynight_log_ratio', 'day_cv_activity', 'short_bout_frac', 'max_activity', 'max_median_ratio']

Ridge Coefficient Magnitudes:
              feature  importance
2          ibi_median    2.015671
3         bout_len_cv    0.710073
6     day_cv_activity    0.634298
8        max_activity    0.352035
9    max_median_ratio    0.352035
4     day_night_ratio    0.167161
5  daynight_log_ratio    0.136717
7     short_bout_frac    0.021398
0        min_activity    0.000000
1     median_activity    0.000000



Leave-2-Out Cross-Validation (Ridge):
Mean Absolute Error: 1.76 ranks
Std: 1.03

Actual vs Predicted Ranks (Ridge):
              animal  actual_rank  ridge_rank
animal_15  animal_15            1           1
animal_11  animal_11            2           2
animal_3    animal_3            3           3
animal_13  animal_13            4           4
animal_5    animal_5            5           5
animal_9    animal_9            6           6
animal_8    animal_8            7           8
animal_12  animal_12            8           7


In [9]:
output_path = "./test_results_2026/feature_importance.html"
rf, mae, all_features, fig_rf = train_rf(all_features, feature_cols, X_scaled, output_path=output_path)

Using top-10 features: ['min_activity', 'median_activity', 'ibi_median', 'bout_len_cv', 'day_night_ratio', 'daynight_log_ratio', 'day_cv_activity', 'short_bout_frac', 'max_activity', 'max_median_ratio']

Random Forest Feature Importance:
              feature  importance
5  daynight_log_ratio    0.182195
8        max_activity    0.178477
4     day_night_ratio    0.149448
3         bout_len_cv    0.112852
9    max_median_ratio    0.109185
6     day_cv_activity    0.097821
7     short_bout_frac    0.096598
2          ibi_median    0.073424
0        min_activity    0.000000
1     median_activity    0.000000



Leave-2-Out Cross-Validation:
Mean Absolute Error: 2.12 ranks
Std: 1.02

Actual vs Predicted Ranks:
              animal  actual_rank  rf_rank
animal_15  animal_15            1        2
animal_11  animal_11            2        1
animal_3    animal_3            3        3
animal_13  animal_13            4        4
animal_5    animal_5            5        6
animal_9    animal_9            6        5
animal_8    animal_8            7        7
animal_12  animal_12            8        8


In [10]:
output_path = "./test_results_2026/feature_importance_adaboost.html"
ada_model, ada_mae, all_features, fig_ada = train_adaboost(all_features, feature_cols, X_scaled, output_path=output_path)

Using top-10 features: ['min_activity', 'median_activity', 'ibi_median', 'bout_len_cv', 'day_night_ratio', 'daynight_log_ratio', 'day_cv_activity', 'short_bout_frac', 'max_activity', 'max_median_ratio']

AdaBoost Feature Importance (mean over estimators):
              feature  importance
4     day_night_ratio    0.203642
5  daynight_log_ratio    0.184992
8        max_activity    0.168548
9    max_median_ratio    0.165413
7     short_bout_frac    0.097420
3         bout_len_cv    0.095563
6     day_cv_activity    0.048523
2          ibi_median    0.035899
0        min_activity    0.000000
1     median_activity    0.000000



Leave-2-Out Cross-Validation (AdaBoost):
Mean Absolute Error: 2.68 ranks
Std: 1.23

Actual vs Predicted Ranks (AdaBoost):
              animal  actual_rank  ada_rank
animal_15  animal_15            1         1
animal_11  animal_11            2         2
animal_3    animal_3            3         3
animal_13  animal_13            4         4
animal_5    animal_5            5         5
animal_9    animal_9            6         6
animal_8    animal_8            7         7
animal_12  animal_12            8         8


In [11]:
output_path = "./test_results_2026/feature_importance_extratrees.html"
et_model, et_mae, all_features, fig_et = train_extratrees(all_features, feature_cols, X_scaled, output_path=output_path)

Using top-10 features: ['min_activity', 'median_activity', 'ibi_median', 'bout_len_cv', 'day_night_ratio', 'daynight_log_ratio', 'day_cv_activity', 'short_bout_frac', 'max_activity', 'max_median_ratio']

Extra Trees Feature Importance:
              feature  importance
2          ibi_median    0.299036
9    max_median_ratio    0.225231
8        max_activity    0.159047
4     day_night_ratio    0.148477
5  daynight_log_ratio    0.141695
7     short_bout_frac    0.018186
6     day_cv_activity    0.006149
3         bout_len_cv    0.002179
0        min_activity    0.000000
1     median_activity    0.000000



Leave-2-Out Cross-Validation (Extra Trees):
Mean Absolute Error: 1.95 ranks
Std: 0.89

Actual vs Predicted Ranks (Extra Trees):
              animal  actual_rank  et_rank
animal_15  animal_15            1        1
animal_11  animal_11            2        1
animal_3    animal_3            3        3
animal_13  animal_13            4        2
animal_5    animal_5            5        3
animal_9    animal_9            6        4
animal_8    animal_8            7        5
animal_12  animal_12            8        6


In [12]:
output_path = "./test_results_2026/feature_importance_logreg.html"
logreg_model, logreg_acc, all_features, fig_logreg = train_logistic(all_features, feature_cols, X_scaled, output_path=output_path)

Using top-10 features: ['min_activity', 'median_activity', 'ibi_median', 'bout_len_cv', 'day_night_ratio', 'daynight_log_ratio', 'day_cv_activity', 'short_bout_frac', 'max_activity', 'max_median_ratio']

Logistic Regression Feature (Coeff) Magnitudes:
              feature  importance
3         bout_len_cv    3.361817
2          ibi_median    2.634827
6     day_cv_activity    2.545223
4     day_night_ratio    2.467037
5  daynight_log_ratio    2.378803
8        max_activity    2.160040
9    max_median_ratio    2.160040
7     short_bout_frac    1.671289
0        min_activity    0.000000
1     median_activity    0.000000



Leave-2-Out Cross-Validation (Logistic):
Accuracy: 0.00%
Std: 0.000

Actual vs Predicted Ranks (Logistic):
              animal  actual_rank  logreg_rank
animal_15  animal_15            1            1
animal_11  animal_11            2            2
animal_3    animal_3            3            3
animal_13  animal_13            4            4
animal_5    animal_5            5            5
animal_9    animal_9            6            6
animal_8    animal_8            7            7
animal_12  animal_12            8            8


In [13]:



def permutation_spearman_pvalue(y_true, y_pred, n_perm=2000, random_state=42):
    rng = np.random.default_rng(random_state)
    observed_rho = stats.spearmanr(y_true, y_pred)[0]
    perm_rhos = []
    for _ in range(n_perm):
        perm = rng.permutation(y_true)
        perm_rhos.append(stats.spearmanr(perm, y_pred)[0])
    perm_rhos = np.array(perm_rhos)
    p_value = (np.sum(np.abs(perm_rhos) >= abs(observed_rho)) + 1) / (n_perm + 1)
    return observed_rho, p_value


def bootstrap_mae_ci(y_true, y_pred, n_boot=2000, random_state=42, alpha=0.05):
    rng = np.random.default_rng(random_state)
    maes = []
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        maes.append(np.mean(np.abs(y_true[idx] - y_pred[idx])))
    ci_low, ci_high = np.percentile(maes, [100 * (alpha / 2), 100 * (1 - alpha / 2)])
    return float(np.mean(np.abs(y_true - y_pred))), float(ci_low), float(ci_high)


In [14]:
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from models import _select_top_k_features

K = 10  # must match k used in the train_* calls

y_true = all_features['actual_rank'].values
models = {
    'rf_rank': train_rf,
    'gb_rank': train_gb,
    'ridge_rank': train_ridge,
    'ada_rank': train_adaboost,
    'et_rank': train_extratrees, 
    'logreg_rank': train_logistic,
}

# Pre-select the top-K features once (same selection used inside every trainer)
_, X_topk = _select_top_k_features(all_features, feature_cols, X_scaled, K)

cv = LeaveOneOut()
results = []
for name, trainer in models.items():
    model, _, _, _ = trainer(all_features.copy(), feature_cols, X_scaled, show_plot=False, k=K)
    y_pred_cv = cross_val_predict(model, X_topk, y_true, cv=cv, method='predict')
    rho, pval = permutation_spearman_pvalue(y_true, y_pred_cv, n_perm=2000)
    mae, ci_low, ci_high = bootstrap_mae_ci(y_true, y_pred_cv, n_boot=2000)
    results.append((name, rho, pval, mae, ci_low, ci_high))

for name, rho, pval, mae, ci_low, ci_high in results:
    print(f"{name:12s} | rho={rho: .3f}, p={pval: .3f} | MAE={mae: .2f} [{ci_low: .2f}, {ci_high: .2f}]")


Using top-10 features: ['min_activity', 'median_activity', 'ibi_median', 'bout_len_cv', 'day_night_ratio', 'daynight_log_ratio', 'day_cv_activity', 'short_bout_frac', 'max_activity', 'max_median_ratio']

Random Forest Feature Importance:
              feature  importance
5  daynight_log_ratio    0.182195
8        max_activity    0.178477
4     day_night_ratio    0.149448
3         bout_len_cv    0.112852
9    max_median_ratio    0.109185
6     day_cv_activity    0.097821
7     short_bout_frac    0.096598
2          ibi_median    0.073424
0        min_activity    0.000000
1     median_activity    0.000000

Leave-2-Out Cross-Validation:
Mean Absolute Error: 2.12 ranks
Std: 1.02

Actual vs Predicted Ranks:
              animal  actual_rank  rf_rank
animal_15  animal_15            1        2
animal_11  animal_11            2        1
animal_3    animal_3            3        3
animal_13  animal_13            4        4
animal_5    animal_5            5        6
animal_9    animal_9         

In [ ]:

# Save LOO-CV summary to CSV
loo_summary_df = pd.DataFrame(results, columns=['model', 'spearman_rho', 'pval', 'mae', 'ci_low', 'ci_high'])
loo_summary_path = f"./test_results_2026/loo_cv_summary_{gender}.csv"
loo_summary_df.to_csv(loo_summary_path, index=False)
print(f"Saved LOO-CV summary to {loo_summary_path}")
print(loo_summary_df.to_string(index=False))


In [15]:
summary_report_path = f"./test_results_2026/summary_report_{gender}.txt"
generate_summary_report(X_scaled, all_features, y, corr_df, rf, mae, corr_sync, pval_sync, X_topk=X_topk, output_path=summary_report_path)


RANK INFERENCE ANALYSIS SUMMARY

1. BASIC STATISTICS
   Number of animals analyzed: 8
   Rank range: 1 to 8

2. TOP PREDICTIVE FEATURES (by absolute correlation):
   ibi_median               : +0.732 (p=0.0388) *
   hourly_entropy           : -0.690 (p=0.0580) 
   power_8h                 : +0.690 (p=0.0580) 
   hourly_entropy_norm      : -0.690 (p=0.0580) 
   cosinor_amplitude        : +0.619 (p=0.1017) 

5. SUPERVISED LEARNING (Random Forest)
   Mean Absolute Error (Leave-One-Out CV): 1.88 ranks
   R² Score: 0.852

6. ACTIVITY SYNCHRONIZATION
   Correlation with rank: 0.190 (p=0.651)

7. PREDICTED vs ACTUAL RANKS:
   animal  actual_rank  rf_rank  rank_error
animal_15            1        2           1
animal_11            2        1           1
 animal_3            3        3           0
animal_13            4        4           0
 animal_5            5        6           1
 animal_9            6        5           1
 animal_8            7        7           0
animal_12            8  

In [58]:
output_path = "./test_results_2026/rank_comparison_visual.html"
fig_summary = summary_visualization(all_features, output_path=output_path)


✓ Visual comparison saved to ./test_results_2026/rank_comparison_visual.html


In [59]:
output_path = './test_results_2026/accuracy_metrics.csv'
fig_accuracy = generate_methods_comparison(all_features, output_path)


RANK PREDICTION ACCURACY ANALYSIS

ACCURACY METRICS BY METHOD:
--------------------------------------------------------------------------------
             Method  Exact Match (%)  Within ±1 (%)  Within ±2 (%)   MAE  Spearman ρ
      Random Forest             50.0          100.0          100.0 0.500    0.952381
  Gradient Boosting            100.0          100.0          100.0 0.000    1.000000
              Ridge             75.0          100.0          100.0 0.250    0.976190
           AdaBoost            100.0          100.0          100.0 0.000    1.000000
        Extra Trees             25.0           37.5          100.0 1.375    0.951876
Logistic Regression            100.0          100.0          100.0 0.000    1.000000



PER-ANIMAL PREDICTION ANALYSIS:

Animal animal_15 (Actual Rank: 1):
  Random Forest       : Rank 2  ✗ Error: ±1
  Gradient Boosting   : Rank 1  ✓ CORRECT
  Ridge               : Rank 1  ✓ CORRECT
  AdaBoost            : Rank 1  ✓ CORRECT
  Extra Trees         : Rank 1  ✓ CORRECT
  Logistic Regression : Rank 1  ✓ CORRECT

Animal animal_11 (Actual Rank: 2):
  Random Forest       : Rank 1  ✗ Error: ±1
  Gradient Boosting   : Rank 2  ✓ CORRECT
  Ridge               : Rank 2  ✓ CORRECT
  AdaBoost            : Rank 2  ✓ CORRECT
  Extra Trees         : Rank 1  ✗ Error: ±1
  Logistic Regression : Rank 2  ✓ CORRECT

Animal animal_3 (Actual Rank: 3):
  Random Forest       : Rank 3  ✓ CORRECT
  Gradient Boosting   : Rank 3  ✓ CORRECT
  Ridge               : Rank 3  ✓ CORRECT
  AdaBoost            : Rank 3  ✓ CORRECT
  Extra Trees         : Rank 3  ✓ CORRECT
  Logistic Regression : Rank 3  ✓ CORRECT

Animal animal_13 (Actual Rank: 4):
  Random Forest       : Rank 4  ✓ CORRECT
  Gradient Boosting 

In [16]:
from analysis_visualization import export_plots_to_pdf
from ranking_methods import plot_proxy_summary

fig_proxies = plot_proxy_summary(proxy_rows)

figures = [
    ('Feature Correlations with Rank', fig_corr),
    ('Animal Activity Comparison', fig_activity),
    ('PCA of Activity Features', fig_pca),
    ('Activity Cross-Correlation Matrix', fig_cross),
    ('Unsupervised Proxies — Performance Summary', fig_proxies),
    ('Supervised Models — Actual vs Predicted Ranks', fig_summary),
    ('Supervised Models — Accuracy Comparison', fig_accuracy),
    ('Feature Importance — Gradient Boosting', fig_gb),
    ('Feature Importance — Ridge', fig_ridge),
    ('Feature Importance — Random Forest', fig_rf),
    ('Feature Importance — AdaBoost', fig_ada),
    ('Feature Importance — Extra Trees', fig_et),
    ('Feature Importance — Logistic Regression', fig_logreg),
]

pdf_path = f"./test_results_2026/report_{gender}.pdf"
export_plots_to_pdf(figures, pdf_path, title=f"Dominance Ranking Report — {gender.capitalize()}")


NameError: name 'fig_summary' is not defined